In [27]:
%load_ext autoreload
%autoreload 2
%cd ~/code/entityrepresentations/

/home/morand/code/entityrepresentations


In [28]:
import torch, gc, sys, os, pathlib, logging
import torch.nn.functional as F
import torch.nn as nn
from torch.utils.data import DataLoader

import numpy as np
from jaxtyping import Float
from datasets import load_dataset
import transformer_lens as tl 
from transformer_lens import HookedTransformer
from importlib import reload
from tqdm import tqdm
import matplotlib.pyplot as plt
import circuitsvis as cv

logging.basicConfig(level=logging.INFO)
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
torch.autograd.set_detect_anomaly(True)

# Import our own code
import utils

In [9]:
import NERfromLLM as ner

In [29]:
torch.cuda.empty_cache()
model_name = "phi-2"  # 3B
model_name = "phi-1_5"  # 1.5B
model = utils.load_model(model_name)
dim = model.QK.shape[-1]
print(f"Model dimension is {dim}")


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loaded pretrained model phi-1_5 into HookedTransformer
Model dimension is 2048


# Utils

In [11]:
def scaled_dot_product_attention(query, key, scale=None):
    """Compute scaled dot product attention between all posible pairs of query and key"""
    scale_factor = 1 / np.sqrt(query.size(-1)) if scale is None else scale
    batch, n_seq, d = query.size()
    #expand Q and K to compute all pairwise scores
    key = key.repeat_interleave(n_seq, dim=-2)
    query = query.repeat(1, n_seq, 1)
    #compute scores
    scores = torch.einsum('...bd,...bd->...b', query, key).reshape(batch, n_seq, n_seq) * scale_factor
    #softmax
    # scores = F.softmax(scores, dim=-2)
    scores /= torch.max(scores)
    return scores


## Custom Self Attention
Methods used to compute queries and keys

In [12]:
from NERfromLLM import SelfAttention
# Test Implementation of our custom SelfAttention Class
d = 4
b = 2
seq = 2
attn = SelfAttention(d, k = d, init_identity=True)

#init weights
reps = torch.arange(b*seq).reshape(b, seq, 1).repeat(1, 1, d).float()
print("representation (batch, seq, dim):\n", reps)
print("\nresulting attention patterns:")
print(attn(reps, apply_softmax=False, causal_mask=False))


representation (batch, seq, dim):
 tensor([[[0., 0., 0., 0.],
         [1., 1., 1., 1.]],

        [[2., 2., 2., 2.],
         [3., 3., 3., 3.]]])

resulting attention patterns:
tensor([[[ 0.,  0.],
         [ 0.,  2.]],

        [[ 8., 12.],
         [12., 18.]]], grad_fn=<ViewBackward0>)


# Data

## Tokenizing NER Tags
We will train an attention module, it will need to train on full contexts

## Loading Data

In [13]:
train_dataset, test_dataset, val_dataset = ner.data.load_datasets("CoNLL2003")

In [14]:
train_dataset.tokenize_and_augment(model)
test_dataset.tokenize_and_augment(model)
val_dataset.tokenize_and_augment(model)

100%|██████████████████████████████████████████████████████████████████████████| 2753/2753 [00:03<00:00, 780.49it/s]


In [15]:
batched = train_dataset.get_loader(batch_size=5)

In [16]:
id = np.random.randint(len(train_dataset))
print(id)
item = train_dataset[id]
for key in ["text", "str_tokens"]:
    print(f"{key}: {item[key]}")

# Print the results
for token, tag in zip(item["str_tokens"], item["token_tags"]):
    print(f"{token.replace(' ','_').center(10)} \t| {tag}")

9783
text: 8. Bart Voskamp ( Netherlands ) TVM 53
str_tokens: ['<|endoftext|>', '8', '.', ' Bart', ' V', 'os', 'kamp', ' (', ' Netherlands', ' )', ' TV', 'M', ' 53']
<|endoftext|> 	| 0
    8      	| 0
    .      	| 0
  _Bart    	| 1
    _V     	| 2
    os     	| 2
   kamp    	| 2
    _(     	| 0
_Netherlands 	| 5
    _)     	| 0
   _TV     	| 3
    M      	| 4
   _53     	| 0


In [17]:
#Check attention patterns
id = np.random.randint(len(train_dataset))
item = train_dataset[id]
print(item["str_tokens"])

cv.attention.attention_patterns(tokens = item["str_tokens"], attention=item["pattern"])

['<|endoftext|>', 'Rub', 'in', " '", 's', ' predecessor', ' at', ' the', ' Treasury', ' ,', ' Lloyd', ' B', 'ents', 'en', ' ,', ' was', ' viewed', ' with', ' suspicion', ' by', ' some', ' in', ' the', ' financial', ' markets', ' who', ' thought', ' he', ' had', ' tried', ' to', ' push', ' down', ' the', ' dollar', ' to', ' gain', ' an', ' edge', ' in', ' trade', ' negotiations', ' with', ' Japan', ' .']


## Extract First and last Token Reps

In [18]:
import torch.nn.functional as F

def get_firstLast_reps(model: HookedTransformer, prompts, entities, layer:int):
    """extract model representation of first and last entity mentions
    representations will be extracted at entity mention tokens after reading f"{prompt}{entity}" 
    Args:
        model: HookedTransformer form TransformerLens to extract representations from
        prompts: list of prompts to use for extraction
        entities: list of entities to extract representations from
        layer: layer at which to retreive the representations
    """ 

    dim = model.QK.shape[-1]
    b_size = len(prompts)
    dtype = model.W_U.dtype
    assert len(entities) == b_size
    eos = model.tokenizer.eos_token_id
    
    if layer < 0:
        #baseline, extract embedding only
        hook_name = utils.get_act_name('embed')
    else:        
        hook_name = utils.get_act_name('resid_post', layer=layer)   # get hook name for the layer output 
    
    #tokenize prompts
    prompt_tokens = model.to_tokens(prompts, padding_side='left')
    entity_tokens = [
        model.to_tokens(entity, prepend_bos=False)
        for entity in entities 
        ]

    ent_lengths = [entity.size(-1) for entity in entity_tokens]
    max_len = max(ent_lengths)
    # print(ent_lengths)
    # print(entity_tokens)

    entity_tokens = [F.pad(entity_token,
                        (0, max_len - entity_token.size(-1)), 
                        value=eos) for entity_token in entity_tokens]
    #concatenate prompts and entities, pad to same length with eos tokens
    # print(entity_tokens)
    entity_tokens = torch.vstack(entity_tokens)
    tokens = torch.hstack([prompt_tokens, entity_tokens])
    
    #check that decoder string is correct
    # print(model.tokenizer.decode(tokens.view(-1).cpu().numpy())) # OK !

    ind_first = prompt_tokens.shape[-1]
    row_inds = torch.arange(b_size).unsqueeze(1)
    tok_inds = torch.tensor([[ind_first, ind_first + length - 1] for length in ent_lengths])
    # print(prompt_tokens)
    # print(entity_tokens)
    # print(tok_inds)

    buffer = torch.zeros(b_size, 2, dim, dtype = dtype)       #create buffer where to store representations
    buffer.requires_grad = False
    buffer = buffer.cuda()

    def save_activations(tensor, buffer, row_inds, tok_inds):
        """Save wanted activations in buffer
        Args:
            tensor: the act cache to modify
            buffer (Tensor): the buffer where to store the wanted activations
            row_inds (Tensor): the row indices of the wanted activations
            tok_inds (Tensor): the token indices of the wanted activations
        """
        #just store the wanted activations in the buffer and return unchanged tensor
        buffer[:] = tensor[row_inds, tok_inds,:]
        return tensor
    
    with torch.no_grad():
        model.run_with_hooks(
            tokens,
            return_type=None,
            fwd_hooks= [(hook_name, lambda tensor, hook: save_activations(tensor, buffer, row_inds, tok_inds) )],
        )
    
    return buffer 

def augment_with_firstLast(model, dataset, layer:int, batch_size:int=5, verbose:bool=False):
    """Augment dataset inplace with first and last entity representations
    Args:
        model: HookedTransformer form TransformerLens to extract representations from
        dataset: dataset to augment
        layer: layer at which to retreive the representations
        batch_size: batch size for extraction
    """
    
    for b_ind in tqdm(range(0, len(dataset), batch_size)):
        
        texts = []
        entities = []
        prompts = []

        for i in range(batch_size):
            if b_ind + i >= len(dataset):
                break
            texts.append(train_dataset[b_ind + i]["text"] )
            entities.append(train_dataset[b_ind + i]["entity"] )
            prompts.append(texts[-1].split(entities[i])[0])

        #extract representations
        reps = get_firstLast_reps(model, prompts, entities, layer=layer)

        for i in range(len(texts)):
            dataset[b_ind + i]["first_reps"] = reps[i][0]
            dataset[b_ind + i]["last_reps"] = reps[i][1]

# Match first token of entity mention from tokens of same mention

## Naive Training
In a first attempt, we just optimize the match with regularization, **without any negative examples.**

Train the Dense attn models with our dataset, augmented with representations from first and last tokens.

### Augment with reps

In [19]:
layer = 20
naive = False

if naive: 
    for data in [train_dataset, test_dataset, val_dataset]:
        augment_with_firstLast(model, data, layer=layer, batch_size=5)
        for param in model.parameters():
            param.requires_grad = False
        r = 50
        attn = DenseMatch(dim, k = r).cuda()

### Training

In [20]:

epochs = 5
lr = 1e-3
batch_size = 50
optimizer = torch.optim.Adam(attn.parameters(), lr=lr)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

#train loop

if naive:
    for epoch in range(epochs):
        for batch in tqdm(train_loader):
            first_reps = batch['first_reps'].cuda() #first_reps is a tensor of shape (batch_size, dim)
            last_reps = batch['last_reps'].cuda()   #last_reps is a tensor of shape (batch_size, dim)
            #forward pass
            # match = QK_attn(first_reps, last_reps, query, key)
            match = attn(first_reps, last_reps)
            loss = - torch.mean(match)
            #backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch}, loss: {loss.item()}")

### Tests

In [21]:
if naive:
    #plot Q and K matrices
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(query.detach().cpu().numpy())
    plt.title("Q matrix")
    plt.subplot(1,2,2)
    plt.imshow(key.detach().cpu().numpy())
    plt.title("K matrix")
    plt.colorbar()

## Training with full Attention Matrix
We there train with all other possible matches in the sentence


### Train full attention model

In [30]:
from NERfromLLM.models import SelfAttention, train_attn
################################################
# Hyperparameters
################################################
layer = 20
batch_size = 20
r = 100 #rank of attention matrices

################################################

for param in model.parameters():
    param.requires_grad = False
attn = SelfAttention(dim, k = r).cuda()
hist = []
logging.info("Building loaders...")
train_loader = train_dataset.get_loader(batch_size=batch_size)
val_loader = val_dataset.get_loader(batch_size=30)

INFO:root:Building loaders...


In [32]:
from NERfromLLM.models import train_attn, plot_hist

hist = train_attn(model, layer, attn, train_loader, val_loader, 
                hist = hist, 
                epochs = 1, 
                lr = 1e-2, 
                n_log = 100,
                grad_clip = 5.0,
                pos_weight = 2,
                accumulation_steps=2
                )

plt.title(f"Training Self Attention on layer {layer} of {model_name}")
plot_hist(hist)

INFO:root:Training Self Attention layer for 1 epochs with batch size 20... 
  1%|█▏                                                                             | 8/557 [00:02<02:53,  3.17it/s]


KeyboardInterrupt: 

In [72]:
from Pathlib import Path
# get root dir for saving checkpoints
root_dir = Path("checkpoints")
checkpoints_folder = "checkpoints"
filename = f"AttnLayer_{model_name}_{r}.pth"
#save model
def save_model(model, filename):
    path = pathlib.Path(filename)
    torch.save(model.state_dict(), filename)
    print(f"Model saved at {filename}")
torch.save(attn.state_dict(), filename)
print(f"Model saved at {filename}")

Model saved at AttnLayer_phi-2_100.pth


### Test

In [37]:
from NERfromLLM.models import Cross_Attn

#compare attention patterns
data = train_dataset
data = test_dataset
item = data[np.random.randint(len(data))]
text = item["text"]
tokens = model.to_tokens(text).cuda()
target = item["pattern"].cuda()
str_tokens = model.to_str_tokens(text, padding_side='left')
print(text)
scores = Cross_Attn(model, tokens, layer, attn, 
                    # apply_softmax=False,
                    )
print(scores.shape)
print(target.shape)
cv.attention.attention_patterns(tokens=str_tokens, attention=torch.vstack([scores,target]), )

In New York , Marvin Benard 's two-run homer snapped a tie and Shawn Estes came one out away from his first complete game as the San Francisco Giants beat the Mets 6-4 .
torch.Size([1, 42, 42])
torch.Size([1, 42, 42])


In [38]:
text = "also on Wednesday, a police spokesman said. The bodies of the soldiers were recovered after the concerted efforts of the Avalanche Rescue Teams ( ART )."
text = "The Eiffel Tower is located near the Seine river, in Paris. It was built in 1889 by Gustave Eiffel."
tokens = model.to_tokens(text).cuda()

scores = Cross_Attn(model, tokens, layer, attn, 
                    # apply_softmax=False,
                    )

print(scores.shape)
cv.attention.attention_pattern(tokens=model.to_str_tokens(text), attention=scores.squeeze(0), )

torch.Size([1, 29, 29])


## Evaluation

In [40]:
from NERfromLLM.NERfromLLM import NER_tags_from_scores, count_perf, compute_metrics

data = val_dataset
item = data[np.random.randint(len(data))]
text = item["text"]
tokens = model.to_tokens(item["text"]).cuda()
scores = Cross_Attn(model, tokens, layer, attn, 
                    apply_softmax=False,
                    )
ner_tags = NER_tags_from_scores(scores)

#map target tags to 0,1,2
target = item["token_tags"]

print(text)
for token, tag, gt in zip(model.to_str_tokens(text), ner_tags, target):
    print(f"{token.replace(' ','_').center(10)} \t| {tag} | {gt}")

count_perf(ner_tags, target)

The result leaves Real on 38 points after 16 games , four ahead of Barcelona .
<|endoftext|> 	| 0 | 0
   The     	| 0 | 0
 _result   	| 0 | 0
 _leaves   	| 0 | 0
  _Real    	| 1 | 3
   _on     	| 0 | 0
   _38     	| 0 | 0
 _points   	| 0 | 0
  _after   	| 0 | 0
   _16     	| 0 | 0
  _games   	| 0 | 0
    _,     	| 0 | 0
  _four    	| 0 | 0
  _ahead   	| 0 | 0
   _of     	| 0 | 0
_Barcelona 	| 1 | 3
    _.     	| 0 | 0


(2, 0, 2)

In [41]:
logging.info("Building Test Loader...")
test_loader = test_dataset.get_loader(batch_size=10)

logging.info("Computing metrics on test set...")
compute_metrics(test_loader, model, attn, layer)

INFO:root:Building Test Loader...
INFO:root:Computing metrics on test set...
  0%|                                                                                       | 0/261 [00:00<?, ?it/s]


NameError: name 'utils' is not defined

# Tests
Quick tests of the functions defined above

### Performance of `compute_to_layer()`

In [99]:
import time
from utils import compute_to_layer

#test compute_to_layer
tokens = model.to_tokens(["Hugging Face is a French company based in New York","It is known for its main product, the hugging face transformer."], padding_side='left')
print(tokens.shape)
layer = 1

hook_name = tl.utils.get_act_name('resid_post', layer=layer)   # get hook name for the layer output 
    
#compute time for the layer
beg = time.perf_counter()
reps = compute_to_layer(model, layer, tokens, verbose = True)
print(f"Time to compute layer {layer} :\n - with compute_to_layer: {time.perf_counter() - beg}")

#do the same for the whole model
beg = time.perf_counter()
_ , cache = model.run_with_cache(tokens)
reps_whole = cache[hook_name].cpu() #shape (batch, len, dim)
print(f" - with whole model: {time.perf_counter() - beg}")

#check that the two computations are the same
print("Are the two computations the same? ", torch.allclose(reps, reps_whole))

ImportError: cannot import name 'compute_to_layer' from 'utils' (/home/morand/code/entityrepresentations/utils.py)

### `scaled_dot_product_attention`

In [15]:
# test scaled_dot_product_attention
batch, n_seq, d = 1, 5, 3
K = torch.arange(n_seq*d).reshape(batch, n_seq, d).float() #shape (batch, n_seq, d)
Q = torch.arange(n_seq*d).reshape(batch, n_seq, d).float()
print(K)
print(Q)
scores = scaled_dot_product_attention(Q, K)
plt.imshow(scores.squeeze().detach().cpu().numpy())
plt.colorbar()

In [16]:
# Demo of torch advanced indexing
a = torch.randn(3, 4, 1)
indexes = torch.tensor([ #indexes to extract from a at each row
    [0, 2],
    [0, 3],
    [1, 1]
    ])
row_indices = torch.arange(len(indexes)).unsqueeze(1)

# Gather elements using advanced indexing
print(a)
print(a[row_indices, indexes, :])

## Run model up to a given layer

In [ ]:
import torch
from transformer_lens import HookedTransformer

# Load your model
model = HookedTransformer.from_pretrained('gpt2')

# Define the custom forward function
def forward_up_to_layer(model, inputs, layer):
    # Tokenize the input
    tokens = model.tokenizer(inputs, return_tensors='pt')
    input_ids = tokens['input_ids']
    
    # Get the embeddings
    hidden_states = model.embeddings(input_ids)
    
    # Pass through the layers up to the specified layer
    for i in range(layer + 1):
        hidden_states = model.blocks[i](hidden_states)
    return hidden_states


In [ ]:
# Define your input
input_text = "Hello, world!"
layer_number = 5

# Run the custom forward pass
with torch.no_grad():
    activations = forward_up_to_layer(model, input_text, layer_number)
# check with whole model
out, cache = model.run_with_cache(input_text)
hook = tl.utils.get_act_name('resid_post', layer=layer_number)
print(torch.allclose(activations, cache[hook]))

# Print the shape of the activations
print(f"Activations up to layer {layer_number}: {activations.shape}")